#### LLM + Symbolic Verifier for SQLi Confirmation

This notebook implements the pipeline shown in your architecture:

1. Load scanner output artifacts (`zap_alerts.json` and `zap_scan_traffic.json`)

2. Use Google AI Studio (Gemini) as a response analyzer to triage SQLi candidates

3. Run deterministic symbolic checks (Z3) to confirm or reject each candidate

4. Output only confirmed SQL injection findings

In [10]:
import json
import os
import re
from pathlib import Path
from urllib.parse import parse_qs, urlparse, unquote_plus
import pandas as pd
from dotenv import load_dotenv

# Resolve .env robustly for both notebook-dir and repo-root execution.
env_candidates = [Path.cwd() / ".env", Path.cwd().parent / ".env"]
loaded_env_path = None
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(dotenv_path=candidate, override=True)
        loaded_env_path = candidate
        break

print(f"Loaded .env from: {loaded_env_path if loaded_env_path else 'not found'}")
print(
    "API key vars detected:",
    bool((os.getenv("GOOGLE_API_KEY") or "").strip()),
    bool((os.getenv("GEMINI_API_KEY") or "").strip()),
)

# Point to your scanner artifacts (works whether cwd is repo root or notebook dir)
BASE_DIR = Path.cwd() / "rule_based_tool"
if not BASE_DIR.exists():
    BASE_DIR = Path.cwd().parent / "rule_based_tool"

ALERTS_PATH = BASE_DIR / "zap_alerts.json"
TRAFFIC_PATH = BASE_DIR / "zap_scan_traffic.json"

print(f"Artifacts dir: {BASE_DIR}")
print(f"Alerts exists: {ALERTS_PATH.exists()}")
print(f"Traffic exists: {TRAFFIC_PATH.exists()}")

with open(ALERTS_PATH, "r", encoding="utf-8") as f:
    alerts = json.load(f)

with open(TRAFFIC_PATH, "r", encoding="utf-8") as f:
    traffic = json.load(f)

print(f"Loaded alerts: {len(alerts)}")
print(f"Loaded traffic messages: {len(traffic)}")

Loaded .env from: c:\Users\Administrator\Projects\neurosymbol_exp\.env
API key vars detected: True False
Artifacts dir: c:\Users\Administrator\Projects\neurosymbol_exp\rule_based_tool
Alerts exists: True
Traffic exists: True
Loaded alerts: 9
Loaded traffic messages: 178


In [11]:
SQLI_ALERT_KEYWORDS = (
    "sql injection",
    "sqli",
    "sql",
)



def get_request_line(request_header: str) -> str:
    return request_header.splitlines()[0] if request_header else ""

def extract_url_from_request_line(request_line: str) -> str:
    parts = request_line.split()
    if len(parts) >= 2:
        return parts[1]
    return ""



def response_signal(response_header: str, response_body: str) -> dict:
    hdr = (response_header or "").lower()
    body = (response_body or "").lower()
    sql_errors = [
        "sql syntax",
        "sqlite",
        "mysql",
        "postgresql",
        "odbc",
        "unterminated",
        "syntax error",
        "warning",
    ]

    matched = [token for token in sql_errors if token in hdr or token in body]
    status_match = re.search(r"HTTP/\d\.\d\s+(\d{3})", response_header or "")
    status_code = int(status_match.group(1)) if status_match else None

    return {

        "status_code": status_code,
        "sql_error_tokens": matched,
        "body_len": len(response_body or ""),

    }



def extract_candidates(messages: list[dict]) -> list[dict]:

    candidates = []
    suspicious_re = re.compile(r"('|\"|--|/\*|\*/|\bOR\b|\bAND\b|\bUNION\b|\bSELECT\b)", re.IGNORECASE)

    for msg in messages:

        request_line = get_request_line(msg.get("requestHeader", ""))

        url = extract_url_from_request_line(request_line)

        if not url:
            continue



        parsed = urlparse(url)
        query_map = parse_qs(parsed.query, keep_blank_values=True)
        decoded_qs = {k: [unquote_plus(vv) for vv in v] for k, v in query_map.items()}

        suspicious_params = {}

        for key, vals in decoded_qs.items():
            hit_vals = [v for v in vals if suspicious_re.search(v)]
            if hit_vals:
                suspicious_params[key] = hit_vals

        if not suspicious_params:
            continue



        sig = response_signal(msg.get("responseHeader", ""), msg.get("responseBody", ""))
        candidates.append(

            {

                "message_id": msg.get("id"),
                "request_line": request_line,
                "url": url,
                "suspicious_params": suspicious_params,
                "response_status": sig["status_code"],
                "sql_error_tokens": sig["sql_error_tokens"],
                "response_body_len": sig["body_len"],
                "response_header": msg.get("responseHeader", ""),
                "response_body": msg.get("responseBody", "")[:3500],

            }

        )



    return candidates



candidates = extract_candidates(traffic)

print(f"Potential SQLi candidates from traffic: {len(candidates)}")

pd.DataFrame(candidates)[["message_id", "response_status", "suspicious_params"]].head(10)

Potential SQLi candidates from traffic: 140


,message_id,response_status,suspicious_params
0,3298,200,{'id': [''']}
1,3299,200,{'id': ['1'']}
2,3300,200,"{'id': ['""']}"
3,3301,200,"{'id': ['1""']}"
4,3304,200,{'id': [''(']}
5,3305,200,{'id': ['1'(']}
6,3312,200,"{'id': [''""']}"
7,3313,200,"{'id': ['1'""']}"
8,3320,200,{'id': ['1 AND 1=1 -- ']}
9,3321,200,{'id': ['1 AND 1=2 -- ']}


In [12]:
google_key = (os.getenv("GOOGLE_API_KEY") or "").strip()
gemini_key = (os.getenv("GEMINI_API_KEY") or "").strip()

API_KEY = google_key or gemini_key
API_KEY_SOURCE = "GOOGLE_API_KEY" if google_key else ("GEMINI_API_KEY" if gemini_key else "NONE")

print(f"API key available: {bool(API_KEY)} (source: {API_KEY_SOURCE})")

API key available: True (source: GOOGLE_API_KEY)


In [19]:
import json
import os
import time
import google.generativeai as genai

API_KEY = (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY") or "").strip()
MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-2.0-flash").strip()
MAX_RETRIES = int(os.getenv("GEMINI_MAX_RETRIES", "2"))
MIN_REQUEST_INTERVAL_SEC = float(os.getenv("GEMINI_MIN_REQUEST_INTERVAL_SEC", "0.5"))
MAX_LLM_CANDIDATES = int(os.getenv("MAX_LLM_CANDIDATES", "20"))


def get_supported_generate_models() -> list[str]:
    """Return model names that support generateContent."""
    names: list[str] = []
    for model in genai.list_models():
        methods = getattr(model, "supported_generation_methods", []) or []
        if "generateContent" in methods:
            names.append(model.name.replace("models/", ""))
    return sorted(set(names))


def resolve_model_name(preferred: str) -> str:
    """Pick a valid model name for current API/key permissions."""
    supported = get_supported_generate_models()

    if not supported:
        raise RuntimeError(
            "No models with generateContent support were returned by API. "
            "Check API key, project, and region permissions."
        )

    if preferred in supported:
        return preferred

    fallback_order = [
        "gemini-2.0-flash",
        "gemini-1.5-flash",
        "gemini-1.5-pro",
    ]
    for name in fallback_order:
        if name in supported:
            return name

    return supported[0]


def is_quota_error(message: str) -> bool:
    msg = (message or "").lower()
    return (
        "resource_exhausted" in msg
        or "quota" in msg
        or "limit: 0" in msg
        or "rate limit" in msg
        or "429" in msg
    )


def is_retryable_llm_error(message: str) -> bool:
    msg = (message or "").lower()
    retry_markers = [
        " 429 ",
        "resource_exhausted",
        "quota",
        "rate limit",
        "temporarily unavailable",
        "service unavailable",
        "deadline exceeded",
        "internal",
    ]
    return any(marker in msg for marker in retry_markers)


def candidate_risk_score(candidate: dict) -> int:
    payload_values = []
    for vals in candidate.get("suspicious_params", {}).values():
        payload_values.extend(vals)
    payload = " | ".join(payload_values).lower()
    score = 0
    score += 3 if "union" in payload else 0
    score += 2 if (" or " in payload or " and " in payload) else 0
    score += 2 if ("--" in payload or "/*" in payload) else 0
    score += 2 if candidate.get("sql_error_tokens") else 0
    score += 1 if (candidate.get("response_status") or 0) >= 500 else 0
    return score


def generate_with_retry(model: genai.GenerativeModel, prompt: str, model_name: str) -> str:
    last_error = ""
    for attempt in range(MAX_RETRIES + 1):
        try:
            return (model.generate_content(prompt).text or "").strip()
        except Exception as ex:
            last_error = str(ex)
            if attempt >= MAX_RETRIES or not is_retryable_llm_error(last_error):
                break
            backoff = 2 ** attempt
            print(
                f"Transient LLM error on {model_name} (attempt {attempt + 1}/{MAX_RETRIES + 1}): "
                f"retrying in {backoff}s"
            )
            time.sleep(backoff)

    raise RuntimeError(last_error or "Unknown LLM failure")


def llm_triage_candidate(candidate: dict, model: genai.GenerativeModel, model_name: str) -> dict:
    prompt = f"""
You are a security response analyzer.
Decide if this HTTP exchange is a likely SQL Injection attempt.

Return ONLY valid JSON with keys:
- likely_sqli (boolean)
- confidence (one of: low, medium, high)
- reason (short string)
- payload_fragment (short string)
- requires_symbolic_check (boolean)

Request line:
{candidate['request_line']}

Suspicious params:
{json.dumps(candidate['suspicious_params'], ensure_ascii=False)}

Response status:
{candidate['response_status']}

Observed SQL error tokens:
{candidate['sql_error_tokens']}

Response header (trimmed):
{candidate['response_header'][:900]}

Response body sample (trimmed):
{candidate['response_body'][:1800]}
"""

    try:
        raw = generate_with_retry(model, prompt, model_name)
        if MIN_REQUEST_INTERVAL_SEC > 0:
            time.sleep(MIN_REQUEST_INTERVAL_SEC)
    except Exception as ex:
        msg = str(ex)
        return {
            "llm_likely_sqli": False,
            "llm_confidence": "low",
            "llm_reason": f"LLM call failed on {model_name}: {msg[:220]}",
            "llm_payload_fragment": "",
            "llm_requires_symbolic_check": True,
        }

    raw = raw.replace("```json", "").replace("```", "").strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {
            "likely_sqli": False,
            "confidence": "low",
            "reason": f"Model response was non-JSON: {raw[:200]}",
            "payload_fragment": "",
            "requires_symbolic_check": True,
        }

    return {
        "llm_likely_sqli": bool(parsed.get("likely_sqli", False)),
        "llm_confidence": str(parsed.get("confidence", "low")),
        "llm_reason": str(parsed.get("reason", "")),
        "llm_payload_fragment": str(parsed.get("payload_fragment", "")),
        "llm_requires_symbolic_check": bool(parsed.get("requires_symbolic_check", True)),
    }


triaged = []

if not API_KEY:
    print("Missing GOOGLE_API_KEY/GEMINI_API_KEY; using symbolic-only mode.")
    for cand in candidates:
        triaged.append(
            {
                **cand,
                "llm_likely_sqli": False,
                "llm_confidence": "low",
                "llm_reason": "Missing API key; using symbolic-only mode.",
                "llm_payload_fragment": "",
                "llm_requires_symbolic_check": True,
            }
        )
else:
    genai.configure(api_key=API_KEY)
    active_model_name = resolve_model_name(MODEL_NAME)
    print(f"Requested model: {MODEL_NAME}")
    print(f"Active model: {active_model_name}")
    print(f"Retries: {MAX_RETRIES}, min interval: {MIN_REQUEST_INTERVAL_SEC}s, max LLM candidates: {MAX_LLM_CANDIDATES}")

    ranked = sorted(candidates, key=candidate_risk_score, reverse=True)
    llm_batch = ranked[:MAX_LLM_CANDIDATES]
    skipped = ranked[MAX_LLM_CANDIDATES:]

    model = genai.GenerativeModel(active_model_name)
    quota_blocked = False

    for cand in llm_batch:
        triaged_row = {**cand, **llm_triage_candidate(cand, model, active_model_name)}
        triaged.append(triaged_row)

        if is_quota_error(triaged_row.get("llm_reason", "")):
            quota_blocked = True
            break

    if quota_blocked:
        print("Quota/rate limit encountered. Remaining candidates marked for symbolic-only verification.")
        for cand in llm_batch[len(triaged):]:
            triaged.append(
                {
                    **cand,
                    "llm_likely_sqli": False,
                    "llm_confidence": "low",
                    "llm_reason": "Skipped due to API quota/rate limit; symbolic-only fallback.",
                    "llm_payload_fragment": "",
                    "llm_requires_symbolic_check": True,
                }
            )

    for cand in skipped:
        triaged.append(
            {
                **cand,
                "llm_likely_sqli": False,
                "llm_confidence": "low",
                "llm_reason": "Skipped by MAX_LLM_CANDIDATES budget; symbolic-only fallback.",
                "llm_payload_fragment": "",
                "llm_requires_symbolic_check": True,
            }
        )

triage_df = pd.DataFrame(triaged)
print(f"Triaged candidates: {len(triage_df)}")
if not triage_df.empty:
    display(triage_df[["message_id", "llm_likely_sqli", "llm_confidence", "llm_reason"]].head(15))

Requested model: gemini-2.0-flash
Active model: gemini-2.0-flash
Retries: 2, min interval: 0.5s, max LLM candidates: 20
Transient LLM error on gemini-2.0-flash (attempt 1/3): retrying in 1s
Transient LLM error on gemini-2.0-flash (attempt 2/3): retrying in 2s
Quota/rate limit encountered. Remaining candidates marked for symbolic-only verification.
Triaged candidates: 140


,message_id,llm_likely_sqli,llm_confidence,llm_reason
0,3341,False,low,LLM call failed on gemini-2.0-flash: 429 You e...
1,3342,False,low,Skipped due to API quota/rate limit; symbolic-...
2,3343,False,low,Skipped due to API quota/rate limit; symbolic-...
3,3344,False,low,Skipped due to API quota/rate limit; symbolic-...
4,3345,False,low,Skipped due to API quota/rate limit; symbolic-...
5,3346,False,low,Skipped due to API quota/rate limit; symbolic-...
6,3399,False,low,Skipped due to API quota/rate limit; symbolic-...
7,3400,False,low,Skipped due to API quota/rate limit; symbolic-...
8,3401,False,low,Skipped due to API quota/rate limit; symbolic-...
9,3405,False,low,Skipped due to API quota/rate limit; symbolic-...


In [ ]:
try:
    from z3 import And, Bool, BoolVal, Or, Solver
except Exception as ex:
    raise ImportError(
        "Failed to import z3-solver symbols. Install the correct package in this kernel: "
        "`pip uninstall -y z3 && pip install z3-solver` and restart the kernel."
    ) from ex


def symbolic_verify(candidate: dict) -> dict:
    payload_values = []
    for vals in candidate.get("suspicious_params", {}).values():
        payload_values.extend(vals)

    payload = " | ".join(payload_values)
    payload_lower = payload.lower()

    has_quote = BoolVal(("'" in payload) or ('"' in payload))
    has_comment = BoolVal(("--" in payload) or ("/*" in payload) or ("*/" in payload))
    has_union = BoolVal("union" in payload_lower)
    has_tautology = BoolVal((" or " in payload_lower) or (" and " in payload_lower))
    response_has_sql_error = BoolVal(len(candidate.get("sql_error_tokens", [])) > 0)
    response_5xx = BoolVal((candidate.get("response_status") or 0) >= 500)

    looks_injection_payload = Or(has_quote, has_comment, has_union, has_tautology)
    impact_signal = Or(response_has_sql_error, response_5xx)

    vulnerable = Bool("vulnerable")
    s = Solver()
    s.add(vulnerable == Or(And(looks_injection_payload, impact_signal), And(has_union, has_quote), And(has_tautology, has_quote)))

    result = s.check()
    if str(result) == "sat":
        model = s.model()
        confirmed = bool(model[vulnerable])
    else:
        confirmed = False

    reason_parts = []
    if "'" in payload or '"' in payload:
        reason_parts.append("contains quote metachar")
    if "union" in payload_lower:
        reason_parts.append("contains UNION token")
    if " or " in payload_lower or " and " in payload_lower:
        reason_parts.append("contains boolean-logic token")
    if candidate.get("sql_error_tokens"):
        reason_parts.append("response has SQL error-like tokens")
    if (candidate.get("response_status") or 0) >= 500:
        reason_parts.append("response status is 5xx")

    return {
        "symbolic_confirmed": confirmed,
        "symbolic_reason": "; ".join(reason_parts) if reason_parts else "No symbolic evidence",
        "payload_preview": payload[:180],
    }


verified = []
for row in triaged:
    if row.get("llm_likely_sqli", False) or row.get("llm_requires_symbolic_check", True):
        verified.append({**row, **symbolic_verify(row)})

verified_df = pd.DataFrame(verified)
print(f"Verified candidates: {len(verified_df)}")
if not verified_df.empty:
    display(verified_df[["message_id", "llm_likely_sqli", "llm_confidence", "symbolic_confirmed", "symbolic_reason"]].head(20))

ImportError: cannot import name 'And' from 'z3' (c:\Users\Administrator\Projects\neurosymbol_exp\.venv\Lib\site-packages\z3\__init__.py)

In [ ]:
if verified_df.empty:

    print("No candidates to confirm.")

else:

    confirmed_df = verified_df[(verified_df["llm_likely_sqli"] == True) & (verified_df["symbolic_confirmed"] == True)].copy()



    keep_cols = [

        "message_id",

        "request_line",

        "suspicious_params",

        "llm_confidence",

        "llm_reason",

        "symbolic_reason",

        "payload_preview",

    ]



    if confirmed_df.empty:

        print("No SQLi vulnerabilities were fully confirmed by LLM + symbolic verifier.")

    else:

        confirmed_df = confirmed_df[keep_cols].sort_values(by="llm_confidence", ascending=False)

        print(f"Confirmed SQLi vulnerabilities: {len(confirmed_df)}")

        display(confirmed_df)



# Optional: save confirmed findings for downstream tooling

out_path = BASE_DIR / "confirmed_sqli_findings.json"

if "confirmed_df" in locals() and not confirmed_df.empty:

    confirmed_df.to_json(out_path, orient="records", indent=2)

    print(f"Saved confirmed findings to: {out_path}")

In [ ]:
import pandas as pd

try:
    from z3 import And, Bool, BoolVal, Or, Solver
except Exception as ex:
    raise ImportError(
        "z3-solver is not available in this kernel. "
        "Install with: pip uninstall -y z3 ; pip install z3-solver"
    ) from ex


def z3_sqli_experiment(payload: str, status_code: int = 200, sql_error_tokens: list[str] | None = None) -> dict:
    """Evaluate one payload with explicit symbolic predicates."""
    sql_error_tokens = sql_error_tokens or []
    payload_lower = (payload or "").lower()

    has_quote = BoolVal(("'" in payload) or ('"' in payload))
    has_comment = BoolVal(("--" in payload) or ("/*" in payload) or ("*/" in payload))
    has_union = BoolVal("union" in payload_lower)
    has_tautology = BoolVal((" or " in payload_lower) or (" and " in payload_lower))
    response_has_sql_error = BoolVal(len(sql_error_tokens) > 0)
    response_5xx = BoolVal((status_code or 0) >= 500)

    looks_injection_payload = Or(has_quote, has_comment, has_union, has_tautology)
    impact_signal = Or(response_has_sql_error, response_5xx)

    vulnerable = Bool("vulnerable")
    s = Solver()
    s.add(
        vulnerable == Or(
            And(looks_injection_payload, impact_signal),
            And(has_union, has_quote),
            And(has_tautology, has_quote),
        )
    )

    sat = str(s.check()) == "sat"
    confirmed = bool(s.model()[vulnerable]) if sat else False

    return {
        "payload": payload,
        "status_code": status_code,
        "sql_error_tokens": sql_error_tokens,
        "confirmed": confirmed,
        "has_quote": ("'" in payload) or ('"' in payload),
        "has_comment": ("--" in payload) or ("/*" in payload) or ("*/" in payload),
        "has_union": "union" in payload_lower,
        "has_tautology": (" or " in payload_lower) or (" and " in payload_lower),
        "response_has_sql_error": len(sql_error_tokens) > 0,
        "response_5xx": (status_code or 0) >= 500,
    }

ImportError: cannot import name 'And' from 'z3' (c:\Users\Administrator\Projects\neurosymbol_exp\.venv\Lib\site-packages\z3\__init__.py)

In [ ]:
# Try payloads and inspect which symbolic predicates were triggered.
z3_samples = [
    {"payload": "1", "status_code": 200, "sql_error_tokens": []},
    {"payload": "1' OR '1'='1", "status_code": 200, "sql_error_tokens": []},
    {"payload": "1' OR '1'='1 --", "status_code": 500, "sql_error_tokens": ["sql syntax"]},
    {"payload": "1 UNION SELECT username,password FROM users", "status_code": 200, "sql_error_tokens": []},
    {"payload": "abc/*test*/", "status_code": 200, "sql_error_tokens": []},
    {"payload": "x'", "status_code": 500, "sql_error_tokens": []},
]

z3_results = [
    z3_sqli_experiment(
        payload=item["payload"],
        status_code=item["status_code"],
        sql_error_tokens=item["sql_error_tokens"],
    )
    for item in z3_samples
]

z3_df = pd.DataFrame(z3_results)
display(z3_df[
    [
        "payload",
        "confirmed",
        "has_quote",
        "has_comment",
        "has_union",
        "has_tautology",
        "response_has_sql_error",
        "response_5xx",
    ]
])

print("Edit z3_samples to test your own payloads and response conditions.")